# S0000vowa: matching the benchmark to Sorcha at the exact observation time (with light-time)

The n-body **benchmark** propagates every object to a *fixed* epoch (MJD 61642.0) with a synthetic
second detection 30 min later. Sorcha observed **S0000vowa** at its *real* visit times
(`mjd0_utc = 61642.391188`, `mjd1_utc = 61642.408893`), ~0.39 d after the epoch. At |v| = 2.2 deg/day
the object genuinely moves **~0.86 deg** in that gap, so the fixed-epoch benchmark position sits ~0.75 deg
from Sorcha even though the *velocity* already matches (vlam -1.632 vs -1.637).

This notebook closes that gap in two steps and lands on Sorcha to **sub-arcsecond**:

| stage | what | sky sep vs Sorcha |
|---|---|---|
| A | benchmark at the fixed epoch (MJD 61642.0) | 0.746 deg (2686") |
| B | propagate n-body to Sorcha's exact time, **no** light-time (`build_visible`) | ~18.3" |
| C | exact time **+ light-time** (`generate_ephemeris`) | **det0 0.122", det1 0.018"** |

The ~18" stage B leaves is the **light-travel-time** term (Dec-dominated): `build_visible` returns the
*geometric* astrometric position, while Sorcha (and `generate_ephemeris`) include the ~148 s light-time
+ topocentric Rubin (X05) observer + aberration. Stage C is the same path that passed the pre-registered
acceptance test (fixing_integrator.md §9.9: 18.31" -> 0.127").

---
## Portability

**Stage C (the headline result) runs anywhere** that has `adam_core` + `adam_assist` — they fetch their
own DE440 / de441_n16 ephemerides, so no Hyak data files are needed. All inputs (8 orbital elements +
Sorcha's truth row) are **embedded as literals below**, so no `.s3m` file and no parquet are required.

**Stages A/B additionally need the Hyak stack** (`neomod/src` on the path + Sorcha's ASSIST planetary
kernel `sorcha_cache_2025-07-06/linux_p1550p2650.440`, 98 MB). They **auto-skip** with a message if that
isn't present, so the notebook still runs end-to-end elsewhere (e.g. on arnor).

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
from astropy.coordinates import SkyCoord
from astropy.time import Time
import astropy.units as u

OBJ = 'S0000vowa'

# --- S3M orbit (heliocentric-ecliptic cometary elements; t_p and t_0 are MJD TDB) -------------
# Provenance: neomod/S3Mdata/S0.s3m, row OID == S0000vowa. Embedded so this notebook is portable.
ELEM = dict(q=0.6981738917, e=0.6923929487, i=18.8960007590,
            node=5.1749709212, argperi=236.4460676237,
            t_p=54100.1821299685, t_0=54466.0000000000, H=23.9740000000)

# --- Sorcha's detected values, night 61642 (the truth we compare against) ---------------------
# Provenance: outputs/s3m_linking/case1/sorcha_comparison_case1_nbody_Vband.parquet
SOR = dict(mjd0_utc=61642.391188036, ra0=18.492736355, dec0=-36.588849838,
           mjd1_utc=61642.408892677, ra1=18.452831334, dec1=-36.577106927,
           vlam=-1.631790951, vbeta=1.471257047,
           mean_mag=23.140629429, mean_mag_V=23.742598539)

def sep_arcsec(ra1, dec1, ra2, dec2):
    return SkyCoord(ra1*u.deg, dec1*u.deg).separation(SkyCoord(ra2*u.deg, dec2*u.deg)).arcsec

# --- optional: the Hyak stack for stages A/B (auto-detected) ----------------------------------
HYAK = Path('/mmfs1/gscratch/dirac/ds2004/sorcha')
HAVE_LOCAL = (HYAK / 'neomod' / 'src').is_dir() and \
             (HYAK / 'sorcha_cache_2025-07-06' / 'linux_p1550p2650.440').is_file()
if HAVE_LOCAL:
    sys.path.insert(0, str(HYAK / 'neomod' / 'src'))

print(f'object   : {OBJ}   q={ELEM["q"]:.4f} e={ELEM["e"]:.4f} i={ELEM["i"]:.3f} H_V={ELEM["H"]:.3f}')
print(f'Sorcha d0: MJD_UTC={SOR["mjd0_utc"]:.6f}  RA/Dec={SOR["ra0"]:.6f}/{SOR["dec0"]:.6f}')
print(f'Sorcha d1: MJD_UTC={SOR["mjd1_utc"]:.6f}  RA/Dec={SOR["ra1"]:.6f}/{SOR["dec1"]:.6f}')
print(f'\nHyak stack present (stages A/B): {HAVE_LOCAL}')

## Stages A & B — fixed epoch, then the exact time (no light-time)

*Hyak-only* (needs `velocity_density_pipeline_gmm` + Sorcha's ASSIST kernel). Skips gracefully elsewhere.

**A** is what the benchmark parquet stores (object at MJD 61642.0) — off by the 0.39 d epoch gap.
**B** propagates the same n-body orbit to each real Sorcha visit time; the gap disappears and the
residual drops to ~18", Dec-dominated — that's the light-travel term B omits.

In [ ]:
geom = {}
if HAVE_LOCAL:
    import velocity_density_pipeline_gmm as vdp
    import neoscore as nsc
    scorer = nsc.NEOMODScorer(None, None, None, None, None)  # only _get_earth_and_observer is used
    obj_df = pd.DataFrame([{**ELEM, 'OID': OBJ, 'a': ELEM['q']/(1.0-ELEM['e'])}])

    def geom_radec(mjd_utc):
        """n-body GEOMETRIC (astrometric, no light-time) RA/Dec at a UTC instant."""
        t = Time(float(mjd_utc), format='mjd', scale='utc')
        v = vdp.build_visible_subset_dataframe(
            obj_df, obstime_str=t, scorer=scorer, max_sep_deg=180.0, chunk=1,
            show_progress=False, center_mode='custom_ecliptic',
            center_lon_deg=0.0, center_lat_deg=0.0)
        p = v.iloc[0]
        return float(p.ra_deg), float(p.dec_deg)

    # A: fixed benchmark epoch
    ra_f, dec_f = geom_radec(61642.0)
    sA = sep_arcsec(ra_f, dec_f, SOR['ra0'], SOR['dec0'])
    print(f'A  fixed epoch 61642.0 : RA/Dec = {ra_f:.6f}/{dec_f:.6f}')
    print(f'   vs Sorcha det0      : sep = {sA:.1f}" = {sA/3600:.3f} deg   <- the epoch gap\n')

    # B: exact Sorcha times, still no light-time
    for det, mjd, sra, sdec in [('det0', SOR['mjd0_utc'], SOR['ra0'], SOR['dec0']),
                                ('det1', SOR['mjd1_utc'], SOR['ra1'], SOR['dec1'])]:
        ra, dec = geom_radec(mjd)
        geom[det] = (ra, dec)
        print(f'B  {det} @ {mjd:.6f}: RA/Dec = {ra:.6f}/{dec:.6f}   '
              f'dRA={(ra-sra)*3600:+.2f}" dDec={(dec-sdec)*3600:+.2f}"  '
              f'sep = {sep_arcsec(ra,dec,sra,sdec):.2f}"')
else:
    print('SKIPPED — stages A/B need the Hyak stack (neomod/src + Sorcha ASSIST kernel).')
    print('Reference values measured on Hyak:  A = 2686" (0.746 deg),  B = 18.31" / 18.27"')

## Stage C — exact times **+ light-time correction** (`generate_ephemeris`)

**This is the portable cell — it runs anywhere with `adam_core` + `adam_assist`.**

Full apparent-place ephemeris: ASSIST n-body + ~148 s light-time + topocentric Rubin (X05) + aberration —
Sorcha's own convention. Two adam_core traps are handled: `tp` is **MJD in the same scale as `time`**
(TDB, *not* JD), and `Orbits` requires **Cartesian** coordinates (handing it `CometaryCoordinates`
silently yields a null state and NaN light-time).

In [ ]:
from adam_core.coordinates import CometaryCoordinates, Origin
from adam_core.orbits import Orbits
from adam_core.orbits.physical_parameters import PhysicalParameters
from adam_core.observers import Observers
from adam_core.time import Timestamp
from adam_assist import ASSISTPropagator

coords = CometaryCoordinates.from_kwargs(
    q=[ELEM['q']], e=[ELEM['e']], i=[ELEM['i']],
    raan=[ELEM['node']], ap=[ELEM['argperi']], tp=[ELEM['t_p']],
    time=Timestamp.from_mjd([ELEM['t_0']], scale='tdb'),
    origin=Origin.from_kwargs(code=['SUN']), frame='ecliptic')
orbits = Orbits.from_kwargs(
    orbit_id=[OBJ], object_id=[OBJ], coordinates=coords.to_cartesian(),
    physical_parameters=PhysicalParameters.from_kwargs(H_v=[ELEM['H']], G=[0.15]))

def apparent_radec(mjd_utc):
    """Apparent RA/Dec: n-body + light-time + X05 topocentric + aberration."""
    obs = Observers.from_code('X05', Timestamp.from_mjd([float(mjd_utc)], scale='utc'))
    e = ASSISTPropagator().generate_ephemeris(orbits, obs, predict_magnitudes=True, max_processes=1)
    return (float(e.coordinates.lon.to_pylist()[0]),
            float(e.coordinates.lat.to_pylist()[0]),
            float(e.light_time.to_pylist()[0]) * 86400.0)

appar = {}
for det, mjd, sra, sdec in [('det0', SOR['mjd0_utc'], SOR['ra0'], SOR['dec0']),
                            ('det1', SOR['mjd1_utc'], SOR['ra1'], SOR['dec1'])]:
    ra, dec, lt = apparent_radec(mjd)
    appar[det] = (ra, dec, lt)
    print(f'C  {det} @ {mjd:.6f}   light_time = {lt:.3f} s')
    print(f'   adam(apparent) = {ra:.6f}/{dec:.6f}    Sorcha = {sra:.6f}/{sdec:.6f}')
    print(f'   sky separation = {sep_arcsec(ra,dec,sra,sdec):.3f}"\n')

## Summary

In [ ]:
rows = []
for det, sra, sdec in [('det0', SOR['ra0'], SOR['dec0']), ('det1', SOR['ra1'], SOR['dec1'])]:
    ra_c, dec_c, lt = appar[det]
    rec = {'detection': det, 'light_time (s)': f'{lt:.1f}'}
    if det in geom:
        ra_b, dec_b = geom[det]
        rec['B: exact-time, no LTC'] = f'{sep_arcsec(ra_b, dec_b, sra, sdec):.2f}"'
    else:
        rec['B: exact-time, no LTC'] = '(skipped)'
    rec['C: exact-time + LTC'] = f'{sep_arcsec(ra_c, dec_c, sra, sdec):.3f}"'
    rows.append(rec)

print('S0000vowa — benchmark vs Sorcha, closing the gap:\n')
print(pd.DataFrame(rows).set_index('detection').to_string())
print('\nA -> B removes the 0.39 d epoch gap (real object motion, ~0.86 deg).')
print('B -> C adds the ~148 s light-time + X05 topocentric observer + aberration.')
print('Result: sub-arcsecond. The benchmark orbit IS Sorcha\'s orbit — the apparent disagreement')
print('was entirely (1) the observation instant and (2) apparent-place corrections.')